# Playground de Langfuse

## Instalación de dependencias

In [1]:
%pip install langchain==0.3.7 langfuse==2.53.3 langchain-google-genai==2.0.4

Note: you may need to restart the kernel to use updated packages.


## Recuperación y ejecución de prompts en cadena

In [ ]:
# Import necessary modules
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv(".env")

# Retrieve API keys from environment variables
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Set environment variables for Langfuse
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")

# Import required classes and functions from langchain and langfuse
from langchain import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import SequentialChain
from langfuse import Langfuse
from langfuse.callback import CallbackHandler

In [16]:
# Initialize Langfuse client
langfuse = Langfuse(
  host="http://localhost:3000"
)

# Initialize Langfuse CallbackHandler for Langchain (tracing)
langfuse_callback_handler = CallbackHandler(
    host="http://localhost:3000",
    user_id="AI Breakfast User",
    session_id="test-20241212"
)

# Get current `production` version of text prompts
langfuse_prompt_1 = langfuse.get_prompt("First-Prompt")
langfuse_prompt_2 = langfuse.get_prompt("Second-Prompt")

# Create instances of the Gemini model
model_1 = ChatGoogleGenerativeAI(
    model=langfuse_prompt_1.config["model"],
    temperature=float(langfuse_prompt_1.config["temperature"]),
)
model_2 = ChatGoogleGenerativeAI(
    model=langfuse_prompt_2.config["model"],
    temperature=float(langfuse_prompt_2.config["temperature"]),
)

# Create ChatPromptTemplates using the Langfuse prompts
langchain_prompt_1 = ChatPromptTemplate.from_template(
        langfuse_prompt_1.get_langchain_prompt(),
        metadata={"langfuse_prompt": langfuse_prompt_1},
    )
langchain_prompt_2 = ChatPromptTemplate.from_template(
        langfuse_prompt_2.get_langchain_prompt(),
        metadata={"langfuse_prompt": langfuse_prompt_2},
    )

# Create LLMChains
first_chain = LLMChain(llm=model_1, prompt=langchain_prompt_1, output_key="content")
second_chain = LLMChain(llm=model_2, prompt=langchain_prompt_2, output_key="post")

# Combine the chains into a SequentialChain
final_chain = SequentialChain(
    chains=[first_chain, second_chain],
    input_variables=["topic", "language"],
    output_variables=["content", "post"],
    verbose=True
)

# Invoke the chain with input variables and Langfuse callback handler
response = final_chain.invoke(
    input = {
        "topic": "Kafka",
        "language": "English"
    },
    config={"callbacks":[langfuse_callback_handler]}
)

# Print the response content (poem)
print(response["post"])



> Entering new SequentialChain chain...

> Finished chain.
**Unleash the Power of Kafka: The Distributed Streaming Platform that's Transforming Data**

Imagine a world where data flows like a river, constantly streaming in from every corner of your business. How do you harness this torrent of information to extract valuable insights and make informed decisions? Enter Apache Kafka, the distributed streaming platform that's revolutionizing the way we manage and process data.

**What is Kafka?**

Think of Kafka as a digital highway, where data producers (like your sensors, applications, and websites) send messages to specific lanes (topics). Data consumers (like your analytics tools, dashboards, and microservices) then subscribe to these lanes to receive the messages they need.

**Why Kafka Rocks:**

* **Lightning-Fast Speed:** Kafka can handle millions of messages per second, ensuring your data is processed in real-time.
* **Unbreakable Durability:** Messages are stored in multiple loc

## Evaluación de prompts

In [16]:
# Import necessary modules
import os
from datetime import datetime


# Import required classes and functions from langchain and langfuse
from langchain import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langfuse import Langfuse

In [22]:
# Define a function to calculate a score (always returns 7)
def p_score_function(output, expected):
  return 7

# Define a function to check if expectations are met (always returns "True")
def es_score_function(output, expected):
  return "True"

# Define a function to provide age rating (always returns "Suitable for all audiences")
def cpe_score_function(output, expected):
  return "Apto para todos los públicos"

# Function to evaluate a single prompt
def evaluation_function(prompt, chain, values, handler):
  generationStartTime = datetime.now()
  # Format the input data
  input_data = prompt.format_prompt(**values).to_string().lstrip("Human:").strip()
  # Generate output using the chain
  output_data = chain.invoke(input=values)[chain.output_key]
  generationEndTime = datetime.now()
 
  # Create a generation object for evaluation
  evaluation = langfuse.generation(
    name="First-Prompt-Evaluation",
    input=input_data,
    output=output_data,
    start_time=generationStartTime,
    end_time=generationEndTime
  )
 
  return output_data, evaluation

# Function to run the experiment on a dataset
def experiment_function(prompt, chain, dataset, experiment_name, handler):
  for item in dataset.items:
    
    # Skip archived items
    if str(item.status) != "DatasetStatus.ARCHIVED":
      # handler = item.get_langchain_handler(run_name=experiment_name) # Keep it
      
      # Evaluate the prompt for this item
      output_data, evaluation = evaluation_function(prompt, chain, item.input, handler)
  
      # Link the evaluation to the item
      item.link(evaluation, experiment_name)
  
      # Add scores to the evaluation
      evaluation.score(
        name="Puntuación",
        value=p_score_function(output_data, item.expected_output),
        comment="Nacho Ojeda Sanchez" # Author not works...
      )
      evaluation.score(
        name="Expectativas satisfechas",
        value=es_score_function(output_data, item.expected_output)
      )
      evaluation.score(
        name="Calificación por edades",
        value=cpe_score_function(output_data, item.expected_output)
      )

In [24]:
# Initialize Langfuse client
langfuse = Langfuse(
  host="http://localhost:3000"
)

# Create a Langfuse callback handler
langfuse_callback_handler = CallbackHandler(
    host="http://localhost:3000",
    user_id="Nacho Ojeda Sanchez",
    session_id="test-20241118"
)

# Retrieve the "First-Prompt" from Langfuse
langfuse_prompt_1 = langfuse.get_prompt("First-Prompt")

# Convert the Langfuse prompt to a LangChain ChatPromptTemplate
langchain_prompt_1 = ChatPromptTemplate.from_template(
        langfuse_prompt_1.get_langchain_prompt(),
        metadata={"langfuse_prompt": langfuse_prompt_1},
    )

# Initialize the Google ChatOpenAI model with parameters from the Langfuse prompt config
model_1 = ChatGoogleGenerativeAI(
    model=langfuse_prompt_1.config["model"],
    temperature=float(langfuse_prompt_1.config["temperature"]),
)

# Set the output key for the LLMChain
output_key = "contenido"

# Create an LLMChain using the model and prompt
first_chain = LLMChain(llm=model_1, prompt=langchain_prompt_1, output_key=output_key)

# Retrieve the "First-Prompt" dataset from Langfuse
dataset_1 = langfuse.get_dataset("First-Prompt")

# Set the experiment name
experiment_name = "test-20241118"

# Run the experiment function with the prepared components
experiment_function(langchain_prompt_1, first_chain, dataset_1, experiment_name, langfuse_callback_handler)